# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates loading, inspecting, and processing the FAIR² colorectal cancer cohort dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library in Python/Jupyter.

### Dataset Source
The dataset schema is described using the [Croissant format](https://mlcommons.org/croissant/) and is available at the following URL:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant pandas

## 1. Data Loading
Load the schema metadata and available records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("\033[1mDataset name:\033[0m", metadata.name)
print("\033[1mDescription:\033[0m", metadata.description)
print("\033[1mVersion:\033[0m", metadata.version)
print("\033[1mLicense:\033[0m", metadata.license)
print("\033[1mTemporal coverage:\033[0m", getattr(metadata, 'temporalCoverage', 'N/A'))

# For further reference, the dataset @id:
print("\033[1mCroissant Dataset @id:\033[0m", metadata.id)


## 2. Data Overview
List all record sets and their fields with their `@id`s, as defined in the Croissant schema.

In [ ]:
# List all record sets and their fields with their @id
record_set_ids = []
print("\033[1mRecord Sets available in the dataset:\033[0m\n")
for record_set in dataset.record_sets:
    print(f"- Record Set name: {record_set.name}")
    print(f"  @id: {record_set.id}\n")
    record_set_ids.append(record_set.id)
    # List fields in this record set
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - Field name: {field.name}")
        print(f"      @id: {field.id}")
    print()

if not record_set_ids:
    print("No record sets found in this dataset's schema.")

## 3. Data Extraction
We'll extract data from the main record set. Use the displayed `@id`s in the previous cell to select a record set and fields.

_Note: If no record sets are printed above, the FAIR² demo schema may require an update, or its distribution is not yet loaded. See the Croissant schema at the URL for details about available data sets._

In [ ]:
# If record sets were discovered above, extract by @id
# For demonstration, let's try to use the first record set if present
if record_set_ids:
    selected_record_set_id = record_set_ids[0]
    print("Loading records from record set @id:", selected_record_set_id)
    # Stream records and load into DataFrame
    records = list(dataset.records(record_set=selected_record_set_id))
    df = pd.DataFrame(records)
    print(f"Loaded {len(df)} records.")
    print("\nAvailable columns (fields by @id):\n", list(df.columns))
    display(df.head())
else:
    print("No record sets found to extract. Please check the schema contents at:", croissant_url)

## 4. Exploratory Data Analysis (EDA)
We'll demonstrate a few typical preprocessing steps using the loaded DataFrame:
* Filtering on a numeric field
* Normalizing that field (z-score)
* Grouping by a categorical field

_Tip: Use the columns listed in the DataFrame above. All column names correspond to Field `@id`s._

In [ ]:
if record_set_ids and not df.empty:
    # Example: try using first numeric field (look for Int/Float-type fields in record set)
    import numpy as np
    selected_numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            selected_numeric_field = col
            break
    if not selected_numeric_field:
        print("No obvious numeric field available for EDA. Please check the DataFrame above.")
    else:
        print(f\
"Using numeric field: {selected_numeric_field} (by @id) for filtering and normalization.\n")
        # Filtering
        threshold = np.percentile(df[selected_numeric_field], 50)
        filtered_df = df[df[selected_numeric_field] > threshold].copy()
        print(f"Filtered for {selected_numeric_field} > {threshold:.2f}; total: {len(filtered_df)} records.")

        # Normalization
        norm_col = f"{selected_numeric_field}_normalized"
        mean = filtered_df[selected_numeric_field].mean()
        std = filtered_df[selected_numeric_field].std() or 1  # avoid division by zero
        filtered_df[norm_col] = (filtered_df[selected_numeric_field] - mean) / std
        print(f"\nFirst 5 rows of normalized '{selected_numeric_field}':")
        display(filtered_df[[selected_numeric_field, norm_col]].head())

        # Example grouping by first non-numeric/categorical field (if any)
        group_field = None
        for col in df.columns:
            if col != selected_numeric_field and not pd.api.types.is_numeric_dtype(df[col]):
                group_field = col
                break
        if group_field:
            print(f"\nGrouping normalized '{selected_numeric_field}' mean by '{group_field}':")
            grouped = filtered_df.groupby(group_field)[[selected_numeric_field, norm_col]].mean()
            display(grouped.head())
        else:
            print("No categorical field to group by found.")
else:
    print("Data was not loaded. Please check earlier steps.")

## 5. Visualization
Let's visualize the distribution of our numeric field (if available) for the subset of records above.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and not df.empty and selected_numeric_field:
    plt.figure(figsize=(8,5))
    sns.histplot(df[selected_numeric_field], kde=True, bins=15)
    plt.title(f"Distribution of {selected_numeric_field} (Field @id)")
    plt.xlabel(selected_numeric_field)
    plt.ylabel("Count")
    plt.show()
    
    # If grouping field was found:
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=df[group_field], y=df[selected_numeric_field])
        plt.title(f"{selected_numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(selected_numeric_field)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()


## 6. Conclusion
In this notebook, we demonstrated how to load, overview, and process a FAIR² record-level biomedical dataset using the `mlcroissant` library.

Key steps shown:
- Loading metadata by Croissant schema URL
- Listing record sets and fields by their `@id`
- Extracting records and mapping to DataFrame columns (fields as `@id`)
- Basic filtering, normalization, summary grouping, and visualization

**Remember:** All entities (record sets, fields) are referenced by their Croissant `@id` as recommended for robust, schema-driven data science. For more, see [MLCommons Croissant documentation](https://mlcommons.github.io/croissant/).

_To go further: adapt the code to select specific record sets/fields, and inspect the full schema at the source URL for advanced analytics or machine learning pipelines!_